In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [ ]:
# Load merged dataset (created from parquet files)
df = pd.read_csv("wl_anova_dataset.csv")

# Make sure these are treated as categorical factors
df["model"] = df["model"].astype("category")
df["tier"] = df["tier"].astype("category")

# Two-way ANOVA with interaction
# WL ~ Model + Tier + Model:Tier
model = ols("wl_similarity ~ C(model) * C(tier)", data=df).fit()

# Type II ANOVA table (good default for balanced-ish designs)
anova = sm.stats.anova_lm(model, typ=2)

print("\n=== Two-way ANOVA (Type II) ===")
print(anova)

# Add partial eta-squared effect size
# partial_eta2 = SS_effect / (SS_effect + SS_error)
ss_error = anova.loc["Residual", "sum_sq"]
anova["partial_eta2"] = anova["sum_sq"] / (anova["sum_sq"] + ss_error)

print("\n=== Effect sizes (partial eta^2) ===")
print(anova[["partial_eta2"]])

# Save ANOVA table
anova.to_csv("anova_wl_model_tier2.csv")
print("\nSaved: anova_wl_model_tier2.csv")

# Optional: model summary (useful for appendix)
with open("anova_model_summary2.txt", "w") as f:
    f.write(model.summary().as_text())
print("Saved: anova_model_summary2.txt")

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("wl_anova_dataset.csv")

# Ensure string types
df["model"] = df["model"].astype(str)
df["tier"] = df["tier"].astype(str)
df["image_filename"] = df["image_filename"].astype(str)

# Single factor for the exact experimental condition
df["config"] = df["model"] + "_" + df["tier"]
df["config"] = df["config"].astype("category")

# Mixed model: random intercept per image
m = smf.mixedlm("wl_similarity ~ C(config)", data=df, groups=df["image_filename"])

res = m.fit(method="lbfgs", reml=False)  # lbfgs often more stable
print(res.summary())

with open("mixedlm_wl_config.txt", "w") as f:
    f.write(res.summary().as_text())
print("Saved: mixedlm_wl_config.txt")

In [ ]:
g = df[df["model"] == "GPT-4.1"].copy()
m = smf.mixedlm("wl_similarity ~ C(tier)", g, groups=g["image_filename"])
res = m.fit(method="lbfgs", reml=False)
print(res.summary())

In [ ]:
import pandas as pd
from scipy.stats import ttest_rel

df = pd.read_csv("wl_anova_dataset.csv")

# Select configurations
gpt_v3 = df[(df["model"] == "GPT-4.1") & (df["tier"] == "v3")]
o4_base = df[(df["model"] == "GPT-o4-mini") & (df["tier"] == "base")]

# Sort to align by image
gpt_v3 = gpt_v3.sort_values("image_filename")
o4_base = o4_base.sort_values("image_filename")

# Sanity check
print(len(gpt_v3), len(o4_base))

assert all(gpt_v3["image_filename"].values == o4_base["image_filename"].values)

# Paired t-test
t_stat, p_val = ttest_rel(gpt_v3["wl_similarity"], o4_base["wl_similarity"])

mean_diff = (gpt_v3["wl_similarity"] - o4_base["wl_similarity"]).mean()

print("\nPaired t-test: GPT-4.1_v3 vs GPT-o4-mini_base")
print("t =", t_stat)
print("p =", p_val)
print("Mean difference =", mean_diff)

In [ ]:
df = pd.read_csv("wl_anova_dataset.csv")

print("Unique models:")
print(df["model"].unique())

print("\nUnique tiers:")
print(df["tier"].unique())